In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)
import keras_tuner as kt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [18]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [19]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [27]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 437s 2s/step - accuracy: 0.4287 - loss: 1.5644 - val_accuracy: 0.5379 - val_loss: 1.2085 - learning_rate: 0.0100
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 412s 2s/step - accuracy: 0.5146 - loss: 1.2597 - val_accuracy: 0.5939 - val_loss: 1.0322 - learning_rate: 0.0100
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 429s 2s/step - accuracy: 0.5509 - loss: 1.1503 - val_accuracy: 0.6165 - val_loss: 0.9989 - learning_rate: 0.0100
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 432s 2s/step - accuracy: 0.5876 - loss: 1.0834 - val_accuracy: 0.5413 - val_loss: 1.1604 - learning_rate: 0.0100
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.5846 - loss: 1.0562 - val_accuracy: 0.5812 - val_loss: 1.0714 - learning_rate: 0.0100
Restoring model weights from the end of the best epoch: 3.


In [29]:
train_loss, lr_train_acc_res = resnet.evaluate(train_ds)
valid_loss, lr_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, lr_test_acc_res = resnet.evaluate(test_ds)
print(lr_train_acc_res)
print(lr_valid_acc_res)
print(lr_test_acc_res)

220/220 ━━━━━━━━━━━━━━━━━━━━ 341s 2s/step - accuracy: 0.6288 - loss: 0.9688
47/47 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.6298 - loss: 0.9969
47/47 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.6254 - loss: 1.0098
0.6288159489631653
0.6298269033432007
0.6254158616065979


In [30]:
res_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
res_results.loc[len(res_results)] = [
    "resnet using SGD",
    lr_train_acc_res,
    lr_test_acc_res,
    lr_valid_acc_res
]
res_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,resnet using SGD,0.628816,0.625416,0.629827


In [31]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [32]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [33]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [34]:
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 379s 2s/step - accuracy: 0.4807 - loss: 1.5898 - val_accuracy: 0.6491 - val_loss: 0.9788 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.5797 - loss: 1.2330 - val_accuracy: 0.5792 - val_loss: 1.2676 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 375s 2s/step - accuracy: 0.5874 - loss: 1.1349 - val_accuracy: 0.5246 - val_loss: 1.3601 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6053 - loss: 1.0683
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 400s 2s/step - accuracy: 0.6053 - loss: 1.0683 - val_accuracy: 0.5373 - val_loss: 1.4433 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.6571 - loss: 0.8664 - val_accuracy: 0.6864 - val_loss: 0.8670 - learning_rate: 5.0000e-04
Restoring model weights from the end of the best epoch: 5.


In [35]:
train_loss, rm_train_acc_res = resnet.evaluate(train_ds)
valid_loss, rm_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, rm_test_acc_res = resnet.evaluate(test_ds)
print(rm_train_acc_res)
print(rm_valid_acc_res)
print(rm_test_acc_res)

220/220 ━━━━━━━━━━━━━━━━━━━━ 336s 2s/step - accuracy: 0.7117 - loss: 0.7800
47/47 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.6911 - loss: 0.8762
47/47 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.6740 - loss: 0.8821
0.7116975784301758
0.6910785436630249
0.6739853620529175


In [36]:
res_results.loc[len(res_results)] = [
    "resnet using RMSprop",
    rm_train_acc_res,
    rm_test_acc_res,
    rm_valid_acc_res
]
res_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,resnet using SGD,0.628816,0.625416,0.629827
1,resnet using RMSprop,0.711698,0.673985,0.691079


In [37]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [38]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [39]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [40]:
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.4789 - loss: 1.5005 - val_accuracy: 0.6039 - val_loss: 1.1035 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 351s 2s/step - accuracy: 0.5757 - loss: 1.1486 - val_accuracy: 0.6272 - val_loss: 1.0213 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 919ms/step - accuracy: 0.6058 - loss: 1.0010 - val_accuracy: 0.6391 - val_loss: 0.9182 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 206s 927ms/step - accuracy: 0.6120 - loss: 0.9518 - val_accuracy: 0.6491 - val_loss: 0.8895 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 922ms/step - accuracy: 0.6404 - loss: 0.8861 - val_accuracy: 0.6391 - val_loss: 1.0044 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 4.


In [41]:
train_loss, adam_train_acc_res = resnet.evaluate(train_ds)
valid_loss, adam_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, adam_test_acc_res = resnet.evaluate(test_ds)
print(adam_train_acc_res)
print(adam_valid_acc_res)
print(adam_test_acc_res)

220/220 ━━━━━━━━━━━━━━━━━━━━ 169s 761ms/step - accuracy: 0.6723 - loss: 0.8209
47/47 ━━━━━━━━━━━━━━━━━━━━ 36s 763ms/step - accuracy: 0.6678 - loss: 0.8809
47/47 ━━━━━━━━━━━━━━━━━━━━ 36s 759ms/step - accuracy: 0.6600 - loss: 0.8820
0.6723252534866333
0.6677762866020203
0.6600133180618286


In [42]:
res_results.loc[len(res_results)] = [
    "resnet using adam",
    adam_train_acc_res,
    adam_test_acc_res,
    adam_valid_acc_res
]
res_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,resnet using SGD,0.628816,0.625416,0.629827
1,resnet using RMSprop,0.711698,0.673985,0.691079
2,resnet using adam,0.672325,0.660013,0.667776


In [43]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [44]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [46]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image)
    return image, label

In [47]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [48]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [49]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [50]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [51]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [52]:
batch_size = 64
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    batch_size = batch_size
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.4628 - loss: 1.5381 - val_accuracy: 0.4874 - val_loss: 1.3559
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.5728 - loss: 1.1185 - val_accuracy: 0.6178 - val_loss: 1.0158
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.5986 - loss: 1.0109 - val_accuracy: 0.6811 - val_loss: 0.8478
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.6193 - loss: 0.9554 - val_accuracy: 0.5719 - val_loss: 1.1590
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.6387 - loss: 0.8902 - val_accuracy: 0.6471 - val_loss: 0.9677


In [53]:
train_loss, bt64_train_acc_res = resnet.evaluate(train_ds)
valid_loss, bt64_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, bt64_test_acc_res = resnet.evaluate(test_ds)
print(bt64_train_acc_res)
print(bt64_valid_acc_res)
print(bt64_test_acc_res)

110/110 ━━━━━━━━━━━━━━━━━━━━ 168s 2s/step - accuracy: 0.6756 - loss: 0.8920
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6378 - loss: 0.9845
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6281 - loss: 1.0117
0.6756062507629395
0.6378162503242493
0.6280771493911743


In [54]:
res_results.loc[len(res_results)] = [
    "resnet using batchsize 64",
    bt64_train_acc_res,
    bt64_test_acc_res,
    bt64_valid_acc_res
]
res_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,resnet using SGD,0.628816,0.625416,0.629827
1,resnet using RMSprop,0.711698,0.673985,0.691079
2,resnet using adam,0.672325,0.660013,0.667776
3,resnet using batchsize 64,0.675606,0.628077,0.637816


In [55]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [56]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [57]:
base_model = resnet.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

resnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_resnet_ft = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 285s 3s/step - accuracy: 0.6014 - loss: 1.2811 - val_accuracy: 0.6531 - val_loss: 0.9355 - learning_rate: 1.0000e-05
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 280s 3s/step - accuracy: 0.6583 - loss: 0.8748 - val_accuracy: 0.6605 - val_loss: 0.8979 - learning_rate: 1.0000e-05
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 275s 2s/step - accuracy: 0.6799 - loss: 0.7838 - val_accuracy: 0.6764 - val_loss: 0.8857 - learning_rate: 1.0000e-05
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 275s 2s/step - accuracy: 0.6930 - loss: 0.6971 - val_accuracy: 0.6904 - val_loss: 0.8498 - learning_rate: 1.0000e-05
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 273s 2s/step - accuracy: 0.7066 - loss: 0.6438 - val_accuracy: 0.6871 - val_loss: 0.8258 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 5.


In [58]:
train_loss, fine_train_acc_res = resnet.evaluate(train_ds)
valid_loss, fine_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, fine_test_acc_res = resnet.evaluate(test_ds)
print(fine_train_acc_res)
print(fine_valid_acc_res)
print(fine_test_acc_res)

110/110 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.7318 - loss: 0.7214
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.7004 - loss: 0.8117
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6873 - loss: 0.8648
0.7318117022514343
0.7003994584083557
0.6872920989990234


In [59]:
res_results.loc[len(res_results)] = [
    "resnet using fine tunning",
    fine_train_acc_res,
    fine_test_acc_res,
    fine_valid_acc_res
]
res_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,resnet using SGD,0.628816,0.625416,0.629827
1,resnet using RMSprop,0.711698,0.673985,0.691079
2,resnet using adam,0.672325,0.660013,0.667776
3,resnet using batchsize 64,0.675606,0.628077,0.637816
4,resnet using fine tunning,0.731812,0.687292,0.700399


In [11]:
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

def build_resnet(hp):
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )
    # 1. Freeze ResNet50 weights for hyperparameter search
    base.trainable = False 

    model = tf.keras.Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(
            hp.Int("units", 64, 128, step=64),
            activation="relu"
        ),
        Dropout(
            hp.Float("dropout", 0.2, 0.5, step=0.1)
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float("learning_rate", 1e-5, 1e-3, sampling="log")
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [12]:
resnet_tuner = kt.RandomSearch(
    build_resnet,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="resnet50"
)
resnet_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 53m 14s]
val_accuracy: 0.5785619020462036

Best val_accuracy So Far: 0.6504660248756409
Total elapsed time: 10h 54m 50s


In [13]:
best_resnet = resnet_tuner.get_best_models(1)[0]

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [14]:
best_hps = resnet_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'units': 128, 'dropout': 0.2, 'learning_rate': 0.00037061779076696727}


In [15]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 635s 3s/step - accuracy: 0.7116 - loss: 0.8003 - val_accuracy: 0.7477 - val_loss: 0.7190
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 613s 3s/step - accuracy: 0.7561 - loss: 0.6763 - val_accuracy: 0.7577 - val_loss: 0.6716
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 644s 3s/step - accuracy: 0.7679 - loss: 0.6325 - val_accuracy: 0.7656 - val_loss: 0.6415
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 634s 3s/step - accuracy: 0.7860 - loss: 0.5803 - val_accuracy: 0.7623 - val_loss: 0.6347
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 729s 3s/step - accuracy: 0.7892 - loss: 0.5592 - val_accuracy: 0.7730 - val_loss: 0.6110


In [17]:
best_resnet.save("cnn_resnet_phase5.keras")

In [18]:
train_loss, hype_train_acc_res = resnet.evaluate(train_ds)
valid_loss, hype_valid_acc_res = resnet.evaluate(valid_ds)
test_loss, hype_test_acc_res = resnet.evaluate(test_ds)
print(hype_train_acc_res)
print(hype_valid_acc_res)
print(hype_test_acc_res)

220/220 ━━━━━━━━━━━━━━━━━━━━ 652s 3s/step - accuracy: 0.8051 - loss: 0.5202
47/47 ━━━━━━━━━━━━━━━━━━━━ 182s 4s/step - accuracy: 0.7823 - loss: 0.5945
47/47 ━━━━━━━━━━━━━━━━━━━━ 126s 3s/step - accuracy: 0.7631 - loss: 0.6522
0.8051355481147766
0.7822902798652649
0.7631403803825378


In [ ]:
res_results.loc[len(res_results)] = [
    "resnet using hyperparameter",
    hype_train_acc_res,
    hype_test_acc_res,
    hype_valid_acc_res
]
res_results

In [ ]:
res_results.to_csv("resnet_comparison.csv",index=False)

In [21]:
resnet.save("cnn_resnet_bestmodel.keras")